In [ ]:
import os
import torch
import numpy as np
from tqdm import tqdm
import yaml
import pandas as pd
from model.dataset import ARCEME_Dataset, get_llto_splits, get_llto_splits_strict, get_val_tiles_auto
from pathlib import Path
from my_utils.check_cubes import generate_exclusion_list
HERE_DIR = Path.cwd() 
ROOT_DIR = HERE_DIR.parent.parent

In [ ]:
# --- 1. Konfiguration für den Test ---
# PROCESSED_DIR = "/net/projects/arceme/vegetation_recovery_prediction/postprocessed"
PROCESSED_DIR = "/scratch/sloeblein/postprocessed"
CSV_PATH = os.path.join(ROOT_DIR, "data_processing", "data","train_test_split.csv")
EXCLUDE_CSV_PATH = os.path.join(ROOT_DIR, "data_processing", "data","excluded_cubes.csv")
K_FOLDS = 3
with open(os.path.join(ROOT_DIR, "model","config.yaml"), "r") as f:
    cfg = yaml.safe_load(f)
PATCH_SIZE = cfg["data"]["patch_size"]
BATCH_SIZE = cfg["training"]["batch_size"]
CONTEXT_LENGTH = cfg["data"]["context_length"]
TARGET_LENGTH = cfg["data"]["target_length"]
v_cfg = cfg["data"]["variables"]

In [ ]:
# df_excluded_cubes = generate_exclusion_list(
#     processed_dir=PROCESSED_DIR, exclude_csv_path=EXCLUDE_CSV_PATH, cfg=cfg
# )
df_excluded_cubes = pd.read_csv(EXCLUDE_CSV_PATH)
df_excluded_cubes
excluded_cube_ids = set(df_excluded_cubes["cube_id"].astype(str).tolist())
# excluded_cube_ids = []

In [ ]:
# --- 2. Splits laden ---
print("🔍 Lade Datensplits...")
all_folds = get_llto_splits(PROCESSED_DIR, CSV_PATH, k=K_FOLDS, show=True, exclude_list=excluded_cube_ids)
train_files, val_files = all_folds[0]

In [ ]:
# --- 2. Splits laden ---
print("🔍 Lade Datensplits...")
all_folds = get_llto_splits_strict(PROCESSED_DIR, CSV_PATH, k=K_FOLDS, show=True, exclude_list=excluded_cube_ids)
train_files, val_files = all_folds[0]

In [ ]:
import pandas as pd
import re
# --- Sanity Check: Ensure no excluded cubes are in the folds ---
if os.path.exists(EXCLUDE_CSV_PATH):
    excluded_df = pd.read_csv(EXCLUDE_CSV_PATH)
    excluded_ids = set(excluded_df["cube_id"].astype(str).tolist())
    
    for f_idx, (train_f, val_f) in enumerate(all_folds):
        # Wir extrahieren die IDs aus den Pfaden (Regex wie im Dataset)
        train_ids = {re.search(r"2\d{3}-\d{4}-[A-Z]{3}", p).group(0) for p in train_f if re.search(r"2\d{3}-\d{4}-[A-Z]{3}", p)}
        val_ids = {re.search(r"2\d{3}-\d{4}-[A-Z]{3}", p).group(0) for p in val_f if re.search(r"2\d{3}-\d{4}-[A-Z]{3}", p)}
        
        bad_in_train = train_ids & excluded_ids
        bad_in_val = val_ids & excluded_ids
        
        if bad_in_train or bad_in_val:
            print(f"❌ FOLD {f_idx} ERROR: Found excluded cubes!")
            if bad_in_train: print(f"   In Train: {bad_in_train}")
            if bad_in_val: print(f"   In Val: {bad_in_val}")
            raise ValueError("Splitting logic failed to exclude bad cubes.")
        else:
            print(f"✅ Fold {f_idx} is clean (No excluded cubes found).")

In [ ]:
fixed_tiles = get_val_tiles_auto(val_files, patch_size=PATCH_SIZE)
len(fixed_tiles)

In [ ]:
print(f"📦 Initialisiere Datasets (Train: {len(train_files)} Cubes, Val: {len(val_files)} Cubes)")
train_ds = ARCEME_Dataset(
    train_files, 
    context_length=CONTEXT_LENGTH, 
    target_length=TARGET_LENGTH, 
    patch_size=PATCH_SIZE, 
    train=True,
    config=cfg,
    exclude_file= excluded_cube_ids,
    s2_vars=v_cfg["s2"],
    s1_vars=v_cfg["s1"],
    era5_vars=v_cfg["era5"],
    static_vars=v_cfg["static"]

)

val_tiles = get_val_tiles_auto(val_files, patch_size=PATCH_SIZE)
val_ds = ARCEME_Dataset(
    val_files, 
    context_length=CONTEXT_LENGTH, 
    target_length=TARGET_LENGTH, 
    patch_size=PATCH_SIZE, 
    train=False, 
    config=cfg,
    exclude_file= excluded_cube_ids,
    s2_vars=v_cfg["s2"],
    s1_vars=v_cfg["s1"],
    era5_vars=v_cfg["era5"],
    static_vars=v_cfg["static"],
    fixed_tiles=val_tiles,
)

In [ ]:
# --- 4. DataLoaders erstellen ---
BATCH_SIZE = 10
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [ ]:
# --- 5. Der eigentliche Test-Loop ---
print("\n🚀 Starte DataLoader-Check (Train-Set)...")

# Wir testen nur die ersten 3 Batches, um Zeit zu sparen
for i, batch in enumerate(train_loader):
    if i >= 3: break
    
    x_ctx, x_fut, y, mask, meta, baseline = batch
    
    print(f"\n--- Batch {i} ---")
    print(f"Shape Context Input (B, T, C, H, W): {x_ctx.shape}")
    print(f"Shape Future Climate:               {x_fut.shape}")
    print(f"Shape Target (kNDVI):               {y.shape}")
    print(f"Shape Target Mask:                  {mask.shape}")
    print(f"Meta:                                {meta}")
    print(f"Shape baseline:                     {baseline.shape}")

    # --- Kritische Prüfungen (Asserts) ---
    
    # 1. Check auf NaNs (Sehr häufiges Problem bei Zarr/NetCDF)
    assert not torch.isnan(x_ctx).any(), f"❌ Fehler: NaNs im Context Input in Batch {i}!"
    assert not torch.isnan(y).any(), f"❌ Fehler: NaNs im Target in Batch {i}!"
    
    # 2. Check auf Datentypen
    assert x_ctx.dtype == torch.float32, f"❌ Fehler: Context ist {x_ctx.dtype}, sollte float32 sein."
    
    # 3. Check auf Wertebereiche (Plausibilität)
    # Beispiel: kNDVI sollte meist zwischen -1 und 1 (oder 0 und 1) liegen
    print(f"Min/Max kNDVI im Batch: {y.min().item():.2f} / {y.max().item():.2f}")
    print(f"Min/Max x_fut im Batch: {x_fut.min().item():.2f} / {x_fut.max().item():.2f}")
    
    # 4. Check ob Maske und Target zusammenpassen
    assert y.shape == mask.shape, "❌ Fehler: Target und Maske haben unterschiedliche Shapes!"

print("\n🚀 Starte DataLoader-Check (Validation-Set mit Tiling)...")
# Ein Batch aus der Validierung testen
val_batch = next(iter(val_loader))
print(f"Validation Batch geladen. Shape: {val_batch[0].shape}")


In [ ]:
# In deinem Notebook, nachdem du den Batch geladen hast:
era5_channels = x_ctx[:, :, 1:1+3, :, :] # Index 1, da kNDVI auf 0 liegt
print(f"ERA5 Range: {era5_channels.min().item():.2f} to {era5_channels.max().item():.2f}")
print(f"ERA5 Mean: {era5_channels.mean().item():.2f}")

In [ ]:
def get_sample_for_cube(dataloader, target_path):
    """Finds the first sample in the dataloader that matches the target path."""
    for batch in dataloader:
        x_ctx, x_future_feat, y_target, target_mask, meta, baseline_sample = batch
        paths = meta['path']
        for b in range(len(paths)):
            if paths[b] == target_path:
                # Wir extrahieren nur dieses eine Sample (Batch-Size 1)
                single_sample = (
                    x_ctx[b:b+1], 
                    x_fut[b:b+1], 
                    y_target[b:b+1], 
                    mask[b:b+1], 
                    {k: [v[b]] if isinstance(v, list) else v[b:b+1] for k, v in meta.items()}
                )
                return single_sample
    return None

In [ ]:
import matplotlib.pyplot as plt
import torch
import numpy as np
import matplotlib.patches as patches

def plot_real_batch_coverage(dataloader, max_batches, target_path, dim_max=1000):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(np.zeros((dim_max, dim_max)), cmap='Greys', alpha=0.1)
    
    patch_count = 0
    for i, batch in enumerate(dataloader):
        if i >= max_batches: break
        meta = batch[4]
        for b in range(len(meta['path'])):
            if meta['path'][b] == target_path:
                rect = patches.Rectangle(
                    (meta['left'][b].item(), meta['top'][b].item()), 
                    dataloader.dataset.patch_size, dataloader.dataset.patch_size,
                    linewidth=1.5, edgecolor='red', facecolor='red', alpha=0.2
                )
                ax.add_patch(rect)
                patch_count += 1

    ax.set_title(f"Spatial Coverage\n{os.path.basename(target_path)}")
    ax.set_xlim(0, dim_max); ax.set_ylim(dim_max, 0)
    plt.show()


def plot_validation_sequence(dataloader, target_cube_idx=0, num_batches_to_show=4):
    """
    Visualisiert, wie der Validierungs-DataLoader den Cube Kachel für Kachel abarbeitet.
    """
    target_path = dataloader.dataset.cube_paths[target_cube_idx]
    patch_size = dataloader.dataset.patch_size
    dim_max = 1000
    
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(np.zeros((dim_max, dim_max)), cmap='Greys', alpha=0.1)
    
    # Farben für die verschiedenen Batches
    colors = plt.cm.viridis(np.linspace(0, 1, num_batches_to_show))
    
    patches_found = 0
    batches_processed = 0
    
    for i, batch in enumerate(dataloader):
        if batches_processed >= num_batches_to_show:
            break
            
        x_ctx, x_future_feat, y_target, target_mask, meta, baseline_sample = batch
        paths = meta['path']
        
        # Prüfen, ob dieser Batch Patches unseres Ziel-Cubes enthält
        batch_has_target = False
        for b in range(len(paths)):
            if paths[b] == target_path:
                batch_has_target = True
                top, left = meta['top'][b].item(), meta['left'][b].item()
                
                rect = patches.Rectangle(
                    (left, top), patch_size, patch_size,
                    linewidth=2, edgecolor=colors[batches_processed], 
                    facecolor=colors[batches_processed], alpha=0.3,
                    label=f"Batch {i}" if b == 0 else ""
                )
                ax.add_patch(rect)
                # Nummerierung der Reihenfolge
                ax.text(left+patch_size/2, top+patch_size/2, str(patches_found), 
                        color='white', ha='center', va='center', fontweight='bold')
                patches_found += 1
        
        if batch_has_target:
            batches_processed += 1

    ax.set_title(f"Validation Sequence for Cube: {os.path.basename(target_path)}\n"
                 f"Showing first {num_batches_to_show} batches of this cube")
    ax.set_xlim(0, dim_max)
    ax.set_ylim(dim_max, 0)
    plt.show()

def visualize_data_content(batch, target_path):
    x_ctx, x_future_feat, y_target, target_mask, meta, baseline_sample = batch
    # Wir suchen das Sample im Batch, das zum Pfad passt
    idx = 0
    for b in range(len(meta['path'])):
        if meta['path'][b] == target_path:
            idx = b
            break

    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    # Context kNDVI (last step)
    axs[0].imshow(x_ctx[idx, -1, 0, :, :].cpu(), cmap='RdYlGn')
    axs[0].set_title(f"Context kNDVI\n{os.path.basename(target_path)}")
    # Target kNDVI (first step)
    axs[1].imshow(y_target[idx, 0, 0, :, :].cpu(), cmap='RdYlGn')
    axs[1].set_title("Target kNDVI")
    # Mask
    axs[2].imshow(mask[idx, 0, 0, :, :].cpu(), cmap='gray')
    axs[2].set_title("Loss Mask")
    plt.show()

def plot_multi_vars_timeseries(batch, target_path, var_indices=[0, 12], var_names=["kNDVI", "t2m_mean"]):
    x_ctx, x_future_feat, y_target, target_mask, meta, baseline_sample = batch
    idx = 0
    for b in range(len(meta['path'])):
        if meta['path'][b] == target_path:
            idx = b
            break

    fig, axs = plt.subplots(len(var_indices), 1, figsize=(12, 4 * len(var_indices)), sharex=True)
    if len(var_indices) == 1: axs = [axs]

    for i, (v_idx, v_name) in enumerate(zip(var_indices, var_names)):
        ctx_val = x_ctx[idx, :, v_idx, :, :].mean(dim=(1, 2)).cpu().numpy()
        if v_name == "kNDVI":
            tar_val = y_target[idx, :, 0, :, :].mean(dim=(1, 2)).cpu().numpy()
        else:
            era5_idx = v_idx - 8 # s2(6) + s1(2)
            tar_val = x_future_feat[idx, :, era5_idx, :, :].mean(dim=(1, 2)).cpu().numpy()

        axs[i].plot(np.arange(len(ctx_val)), ctx_val, color='royalblue', label="Context", marker='.')
        axs[i].plot(np.arange(len(ctx_val), len(ctx_val)+len(tar_val)), tar_val, color='crimson', label="Target", marker='.')
        axs[i].axvline(x=len(ctx_val)-0.5, color='black', ls='--')
        axs[i].set_title(f"Variable: {v_name}")
        axs[i].legend()
    plt.show()


def plot_training_evolution(dataset, target_cube_idx=0, num_epochs=10):
    """Zeigt, welche Patches das Modell über mehrere Epochen von einem Cube sieht."""
    dim_max = 1000
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(np.zeros((dim_max, dim_max)), cmap='Greys', alpha=0.1)
    
    colors = plt.cm.jet(np.linspace(0, 1, num_epochs))
    target_path = dataset.cube_paths[target_cube_idx]
    
    for epoch in range(num_epochs):
        # Wir simulieren den Zugriff in einer Epoche
        _, _, _, _, meta = dataset[target_cube_idx]
        
        rect = patches.Rectangle(
            (meta['left'], meta['top']), 
            dataset.patch_size, dataset.patch_size,
            linewidth=2, edgecolor=colors[epoch], facecolor='none', 
            alpha=0.8, label=f"Epoche {epoch+1}"
        )
        ax.add_patch(rect)
        
    ax.set_title(f"Training Evolution: Zufällige Patches über {num_epochs} Epochen\nCube: {os.path.basename(target_path)}")
    ax.set_xlim(0, dim_max); ax.set_ylim(dim_max, 0)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.show()


In [ ]:
# --- RUNNING THE TESTS ---
TARGET_IDX = 3 # Wähle den 1. Cube aus der Liste
target_path = train_ds.cube_paths[TARGET_IDX]

print(f"🎬 Testing with Cube: {os.path.basename(target_path)}")

# 1. Coverage Test (scannt mehrere Batches)
print("Running Coverage Test...")
plot_real_batch_coverage(train_loader, max_batches=100, target_path=target_path)

# --- Aufruf für die Validation ---
print("Validation coverage")
plot_validation_sequence(val_loader, target_cube_idx=0, num_batches_to_show=4)

# 2. Sample für diesen Cube holen (für die Inhalts-Plots)
print("Searching for a sample of this cube in DataLoader...")
specific_batch = get_sample_for_cube(train_loader, target_path)

if specific_batch:
    print("Running Timeseries Test...")
    plot_multi_vars_timeseries(specific_batch, target_path, var_indices=[0, 12], var_names=["kNDVI", "t2m_mean"])

    print("Running Content Visualizer...")
    visualize_data_content(specific_batch, target_path)
else:
    print("Could not find a sample for this cube in the first few batches. Try increasing the search or use val_loader.")


# Test für den ersten Cube
plot_training_evolution(train_ds, target_cube_idx=0, num_epochs=10)

In [ ]:
def plot_manual_coverage(dataset, target_cube_idx=0, num_patches=5, dim_max=1000):
    """Zieht manuell N Patches aus demselben Cube des Datasets."""
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(np.zeros((dim_max, dim_max)), cmap='Greys', alpha=0.1)
    
    # Wir rufen __getitem__ immer mit demselben Index auf
    # Da train=True ist, wird jedes Mal ein neuer zufälliger Top/Left Offset generiert
    for i in range(num_patches):
        _, _, _, _, meta = dataset[target_cube_idx]
        
        rect = patches.Rectangle(
            (meta['left'], meta['top']), 
            dataset.patch_size, dataset.patch_size,
            linewidth=1.5, edgecolor='red', facecolor='red', alpha=0.2
        )
        ax.add_patch(rect)
    
    target_path = dataset.cube_paths[target_cube_idx]
    ax.set_title(f"Manuelle Abdeckung (5 Zufallscrops)\nCube: {os.path.basename(target_path)}")
    ax.set_xlim(0, dim_max); ax.set_ylim(dim_max, 0)
    plt.show()

# Teste es direkt am Dataset, nicht am Loader!
plot_manual_coverage(train_ds, target_cube_idx=0, num_patches=5)